# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

95141.71s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/redondo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/redondo/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com/"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com/"
os.environ["LANGSMITH_PROJECT"] = f"LangSmith - s07 assignment"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '1d6bc3'. Skipping!
Property 'summary' already exists in node '5c31b9'. Skipping!
Property 'summary' already exists in node '7b7530'. Skipping!
Property 'summary' already exists in node '77f880'. Skipping!
Property 'summary' already exists in node 'f33d89'. Skipping!
Property 'summary' already exists in node '56e773'. Skipping!
Property 'summary' already exists in node '59f822'. Skipping!
Property 'summary' already exists in node '41f6b2'. Skipping!
Property 'summary' already exists in node 'eef4f6'. Skipping!
Property 'summary' already exists in node '16cfea'. Skipping!
Property 'summary' already exists in node 'c20806'. Skipping!
Property 'summary' already exists in node '6f61c8'. Skipping!
Property 'summary' already exists in node '01897d'. Skipping!
Property 'summary' already exists in node '229ab9'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '5c31b9'. Skipping!
Property 'summary_embedding' already exists in node '7b7530'. Skipping!
Property 'summary_embedding' already exists in node '59f822'. Skipping!
Property 'summary_embedding' already exists in node '1d6bc3'. Skipping!
Property 'summary_embedding' already exists in node '56e773'. Skipping!
Property 'summary_embedding' already exists in node '77f880'. Skipping!
Property 'summary_embedding' already exists in node '229ab9'. Skipping!
Property 'summary_embedding' already exists in node 'c20806'. Skipping!
Property 'summary_embedding' already exists in node 'eef4f6'. Skipping!
Property 'summary_embedding' already exists in node 'f33d89'. Skipping!
Property 'summary_embedding' already exists in node '6f61c8'. Skipping!
Property 'summary_embedding' already exists in node '41f6b2'. Skipping!
Property 'summary_embedding' already exists in node '16cfea'. Skipping!
Property 'summary_embedding' already exists in node '01897d'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 480)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 480)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


##### ✅ Answer:

- `SingleHopSpecificQuerySynthesizer`: Creates questions that ask for a specific fact where the answer is located entirely within one single piece of text.
- `MultiHopSpecificQuerySynthesizer`: This creates questions that ask for a specific fact, but you must combine information from multiple different text chunks to find the answer.
- `MultiHopAbstractQuerySynthesizer`: This creates questions that require you to synthesize information from multiple text chunks to form a summary, comparison, or a new conclusion.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the role of the School Participaton Di...,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not provide specific details ...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding ac...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(a) specifies the minimum number o...,single_hop_specifc_query_synthesizer
2,What is the significance of Chapter 3 in relat...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,Is Federal Work-Study a payment period program?,[Non-Term Characteristics A program that measu...,"No, the Federal Work-Study (FWS) Program is an...",single_hop_specifc_query_synthesizer
4,What is the significance of Volume 8 in the co...,[both the credit or clock hours and the weeks ...,Volume 8 relates to the principles of Direct L...,single_hop_specifc_query_synthesizer
5,"According to the regulatory citations, what ar...","[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulatory citations specify that for cred...,multi_hop_abstract_query_synthesizer
6,disbursement timing in subscription programs a...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that in subscription-base...,multi_hop_abstract_query_synthesizer
7,How does the impact of term length variations ...,[<1-hop>\n\nInclusion of Clinical Work in a St...,"The context explains that nonstandard terms, s...",multi_hop_abstract_query_synthesizer
8,Waht is Volume 2 and Volume 7?,"[<1-hop>\n\nChapter 1 Academic Years, Academic...","Based on the context, Volume 2 contains inform...",multi_hop_specific_query_synthesizer
9,How do the disbursement timing rules outlined ...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,The disbursement timing rules in Volume 8 spec...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'b076a7'. Skipping!
Property 'summary' already exists in node 'fcb87c'. Skipping!
Property 'summary' already exists in node '16fc0c'. Skipping!
Property 'summary' already exists in node '530a8a'. Skipping!
Property 'summary' already exists in node '2eda53'. Skipping!
Property 'summary' already exists in node '009be8'. Skipping!
Property 'summary' already exists in node '1a6f72'. Skipping!
Property 'summary' already exists in node '8436e6'. Skipping!
Property 'summary' already exists in node 'dc1c20'. Skipping!
Property 'summary' already exists in node '11b5b1'. Skipping!
Property 'summary' already exists in node '5a25f3'. Skipping!
Property 'summary' already exists in node 'f117f3'. Skipping!
Property 'summary' already exists in node '912c89'. Skipping!
Property 'summary' already exists in node '467eab'. Skipping!


...

Property 'summary_embedding' already exists in node 'b076a7'. Skipping!
Property 'summary_embedding' already exists in node '16fc0c'. Skipping!
Property 'summary_embedding' already exists in node '8436e6'. Skipping!
Property 'summary_embedding' already exists in node '467eab'. Skipping!
Property 'summary_embedding' already exists in node '530a8a'. Skipping!
Property 'summary_embedding' already exists in node '2eda53'. Skipping!
Property 'summary_embedding' already exists in node 'fcb87c'. Skipping!
Property 'summary_embedding' already exists in node '1a6f72'. Skipping!
Property 'summary_embedding' already exists in node '009be8'. Skipping!
Property 'summary_embedding' already exists in node '11b5b1'. Skipping!
Property 'summary_embedding' already exists in node 'dc1c20'. Skipping!
Property 'summary_embedding' already exists in node '5a25f3'. Skipping!
Property 'summary_embedding' already exists in node 'f117f3'. Skipping!
Property 'summary_embedding' already exists in node '912c89'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the department?,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not explicitly define what th...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(b) specify regarding we...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) pertains to weeks of instructi...,single_hop_specifc_query_synthesizer
2,Volume 8 include clinical work?,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,Is Fedral Work-Study a term program?,[Non-Term Characteristics A program that measu...,"No, the Federal Work-Study (FWS) Program is an...",single_hop_specifc_query_synthesizer
4,How do the regulatory citations 34 CFR 668.3(a...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulatory citations 34 CFR 668.3(a) and 3...,multi_hop_abstract_query_synthesizer
5,How do the timing and scheduling constraints o...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The timing and scheduling constraints of clini...,multi_hop_abstract_query_synthesizer
6,If a program includes clinical work that is ou...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work outside standar...,multi_hop_abstract_query_synthesizer
7,H0w doez inclusion of clinical work in a stand...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Inclusion of clinical work in a standard term ...,multi_hop_abstract_query_synthesizer
8,Chapter 2 and Chapter 3 how they relate to dis...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Chapter 2 discusses the inclusion of clinical ...,multi_hop_specific_query_synthesizer
9,Volume 8 include clinical work in standard ter...,[<1-hop>\n\nInclusion of Clinical Work in a St...,"Yes, according to Volume 8, if clinical work m...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (including student Federal PLUS Loans and parent Direct PLUS Loans on behalf of dependent students)  \n- Subsidized and Unsubsidized Federal Stafford Loans  \n- Federal SLS Loans  \n- Federal PLUS Loans  \n\nNote that Subsidized and Unsubsidized Federal Stafford Loans, Federal SLS Loans, and Federal PLUS Loans were made under the Federal Family Education Loan (FFEL) Program, which ended for new loans effective July 1, 2010. New loans are issued under the Direct Loan Program, including Direct Subsidized, Direct Unsubsidized, and Direct PLUS Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

##### ✅ Answer:

- `qa_evaluator`: Evaluates wether the answer is correct or not.
- `labeled_helpfulness_evaluator`: Adds a criteria to evaluate whether the answer is helpful to the user.
- `empathy_evaluator`: Adds a criteria to evaluate whether the answer has empathy and makes the user feel like being understand.

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'passionate-level-72' at:
https://smith.langchain.com/o/cfab6fa1-0fe6-40b2-b083-196767d31fa3/datasets/c45bbc4d-ee51-4e32-ad9e-d6bd4eabed19/compare?selectedSessions=416bbfbf-4521-461f-81c4-9afdb3f1f79f




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Appendices A and B relate to disburseme...,I don't know.,None,Appendix A provides guidance on calculating Pe...,0,0,0,2.142118,8bd4cc32-d844-4e42-9e22-f5c14996a62f,e2688215-dc70-411b-9c8f-666a519d8bc0
1,How do the requirements outlined in Volume 2 a...,"Based on the provided context, the requirement...",None,The requirements in Volume 2 specify that each...,1,1,0,8.759498,834f4d68-2f0a-45e5-b907-5749e329e587,98452c96-3fa6-433b-b241-dd3fe254cfe0
2,Volume 8 include clinical work in standard ter...,"Yes, Volume 8 includes guidance on clinical wo...",None,"Yes, according to Volume 8, if clinical work m...",1,1,0,2.088990,2dd85d34-631d-44aa-afab-e74492e5bae9,7ca0082e-e09a-4034-a4ad-96c375d89702
3,Chapter 2 and Chapter 3 how they relate to dis...,I don't know.,None,Chapter 2 discusses the inclusion of clinical ...,0,0,0,2.559529,1d1ce1b3-b16a-4c27-904e-87c0fc73f760,5f580902-ab59-47b3-a915-31863d0e420c
4,H0w doez inclusion of clinical work in a stand...,I don't know.,None,Inclusion of clinical work in a standard term ...,0,0,0,1.111727,d69d764a-0476-4b15-b8aa-303ec6926d7c,ce4db6cd-2405-4492-856d-5eee152e1b93
5,If a program includes clinical work that is ou...,Based on the provided context:\n\nIf a program...,None,The inclusion of clinical work outside standar...,1,1,0,6.518025,45de6e72-e394-4a87-89f2-1c9fc185ef32,280ebb44-f350-45d3-befc-102d55989d30
6,How do the timing and scheduling constraints o...,I don't know.,None,The timing and scheduling constraints of clini...,0,0,0,1.993546,9e7ca32c-3a94-45de-8dde-7ddd887948a7,283be4e3-6099-4610-b14c-fdf94601ce0e
7,How do the regulatory citations 34 CFR 668.3(a...,The regulatory citations 34 CFR 668.3(a) and 3...,None,The regulatory citations 34 CFR 668.3(a) and 3...,1,1,0,3.414698,cb736144-7fd0-4fd1-b6f7-8c47be0ce26a,89ea7e28-0435-4083-aea4-70b441e02d0e
8,Is Fedral Work-Study a term program?,"Based on the provided context, Federal Work-St...",None,"No, the Federal Work-Study (FWS) Program is an...",1,1,0,2.949961,d4fb074b-0573-4b34-9692-d84a7189b586,7fda44d0-4a56-400f-a73d-54e6ab570b78
9,Volume 8 include clinical work?,"Yes, Volume 8 includes guidance on clinical wo...",None,Inclusion of clinical work in a standard term ...,1,1,0,2.189316,bb29a2b4-0200-4314-9298-66ab942228a9,d2fd85c5-9656-4057-8626-e20171b86432


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

With larger chunks, the retriever is more likely to find chunks that contain complete answers rather than partial information, as it would be the case with smaller chunks.

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Upgrading from `text-embedding-3-small` to `text-embedding-3-large` improves RAG performance because the larger model creates higher-dimensional, more nuanced vector representations that better capture semantic meaning and domain-specific terminology. This leads to more accurate similarity calculations between user queries and document chunks, resulting in better retrieval precision and fewer irrelevant results.

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided in the context, there are several types of loans available:\n\n1. **Direct Subsidized Loans** – These loans are for students who demonstrate financial need, with eligibility tied to the difference between the Cost of Attendance (COA) and the student's financial aid (Standard Aid and Other Financial Aid). They have a maximum eligibility limit based on need.\n\n2. **Direct Unsubsidized Loans** – These are available to students regardless of financial need and can be combined with Direct Subsidized Loans. Additional unsubsidized loan amounts are available if a dependent student’s parent is ineligible for a Direct PLUS Loan.\n\n3. **Direct PLUS Loans** – These loans are available to parents of dependent undergraduate students to help cover the student’s COA, provided the parent meets eligibility requirements. Graduate and professional students can also take Direct PLUS Loans. There is no fixed loan limit, but the amount cannot

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'perfect-lead-54' at:
https://smith.langchain.com/o/cfab6fa1-0fe6-40b2-b083-196767d31fa3/datasets/c45bbc4d-ee51-4e32-ad9e-d6bd4eabed19/compare?selectedSessions=e665265c-e7a2-4871-9e7f-0bcb5c03cbab




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Appendices A and B relate to disburseme...,Thank you for your thoughtful question. Based ...,None,Appendix A provides guidance on calculating Pe...,1,0,1,5.629572,8bd4cc32-d844-4e42-9e22-f5c14996a62f,8aba60c7-8177-4bbd-bb5b-270d6c83ea76
1,How do the requirements outlined in Volume 2 a...,Thank you for your thoughtful question. I can ...,None,The requirements in Volume 2 specify that each...,0,0,1,6.947192,834f4d68-2f0a-45e5-b907-5749e329e587,0bbef724-1861-4ebe-bafd-a23d4f1caedf
2,Volume 8 include clinical work in standard ter...,Thank you for your thoughtful question. Based ...,None,"Yes, according to Volume 8, if clinical work m...",1,1,1,3.889519,2dd85d34-631d-44aa-afab-e74492e5bae9,b70d8415-88c4-4229-812f-e0ee9bb1ffc2
3,Chapter 2 and Chapter 3 how they relate to dis...,Thank you for your thoughtful question about h...,None,Chapter 2 discusses the inclusion of clinical ...,0,0,1,7.933922,1d1ce1b3-b16a-4c27-904e-87c0fc73f760,76b37224-4183-4ed0-a32b-2fdb29cdbe40
4,H0w doez inclusion of clinical work in a stand...,Thank you for your thoughtful question. Based ...,None,Inclusion of clinical work in a standard term ...,1,1,1,6.445366,d69d764a-0476-4b15-b8aa-303ec6926d7c,829d5ed7-9e43-4406-a120-4180a837e78e
5,If a program includes clinical work that is ou...,Thank you for your thoughtful question. It sou...,None,The inclusion of clinical work outside standar...,1,1,1,7.690962,45de6e72-e394-4a87-89f2-1c9fc185ef32,30cdd813-6ad5-488a-8bbc-0bd0aa470543
6,How do the timing and scheduling constraints o...,Thank you for your thoughtful question. Based ...,None,The timing and scheduling constraints of clini...,1,1,1,6.364206,9e7ca32c-3a94-45de-8dde-7ddd887948a7,a0b9da06-e629-4f91-8b6e-890f71b3bfda
7,How do the regulatory citations 34 CFR 668.3(a...,Thank you for your thoughtful question about h...,None,The regulatory citations 34 CFR 668.3(a) and 3...,0,0,1,5.513340,cb736144-7fd0-4fd1-b6f7-8c47be0ce26a,940c2586-a71d-40d1-85f8-d4fbf6a09e27
8,Is Fedral Work-Study a term program?,I see you're asking about whether Federal Work...,None,"No, the Federal Work-Study (FWS) Program is an...",1,1,1,3.294638,d4fb074b-0573-4b34-9692-d84a7189b586,387de4af-d862-4254-a3a6-d2041792fcaa
9,Volume 8 include clinical work?,Thank you for your question. Based on the cont...,None,Inclusion of clinical work in a standard term ...,1,1,1,2.981938,bb29a2b4-0200-4314-9298-66ab942228a9,46562f28-0870-4e0c-8250-84f85b9c4648


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

##### ✅ Answer:

<table>
  <tr>
    <td>Chain 1</td>
    <td>Chain 2</td>
  </tr>
  <tr>
    <td><img src="experiment_one.png"/></td>
    <td><img src="experiment_two.png"/></td>
  </tr>
</table>

Observations:

The most noticiable change is that when we specify in our prompt to answer with empathy, our evaluation for empathy increased dramatically, meaning the model is correctly following the instructions. Also, maybe not as noticiable but also relevant, is that the correctness increased a bit, the cost decreased but the latency doubled, and we can safely assume this is due to the larger chunk size and the change to a large embedding models, because larger chunks means more precise answer given the nature of our corpus, but it also means more computation to apply calculations over this chunks.